# Responsibilities of an AI Engineer

In the previous notebook, you classified job postings and saved the results to `jobs/2-classified-jobs.jsonl`.

In this notebook, you will use an LLM to summarize the responsibilities that appear most often across the AI engineering roles.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

## Load the Classified Jobs

This cell reads the JSONL file created by `4-classify-job-postings.ipynb`.

If the file is missing, run the classification notebook first.

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(classified_jobs_path, lines=True)
print(f"Loaded {len(jobs_for_summary)} classified job postings")

## Keep Only AI Engineering Roles

The classified jobs file contains all postings, including the ones the model rejected.

This cell keeps only the rows where `is_ai_engineering_role` is `True`.

In [ ]:
# Keep only the jobs the LLM classified as true AI engineering roles
jobs_for_summary = jobs_for_summary[jobs_for_summary["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_for_summary)} classified AI engineering job postings")

## Combine All Descriptions

We merge all job descriptions into one large string so we can send them to the model in a single request.

Each description is separated by `---` so the model can tell where one posting ends and the next begins.

In [ ]:
descriptions = jobs_for_summary["description"].astype(str)
job_descriptions = "\n\n---\n\n".join(descriptions)

print(job_descriptions[:10000] + "\n\n...")

## Prepare the Summarization Prompt

We combine the job descriptions into one prompt and ask the model to summarize the responsibilities that appear repeatedly across the postings.

In [ ]:
prompt = f"""
Read the job descriptions below and summarize the responsibilities that appear repeatedly across them.

Return markdown in exactly this format:
### Shared responsibilities
- 5 concise bullet points

Requirements:
- Focus only on common responsibilities
- Ignore benefits, recruiting language, company marketing, and hiring process details
- Keep bullets concrete and specific
- Do not mention company names

Each job description is separated by "---".

Job descriptions:
{job_descriptions}
""".strip()

print("Prompt for summarization: \n")
print(prompt[:1000])
print(f"\nPrompt character count: {len(prompt):,}")

## Count the Prompt Tokens

Before sending the prompt, we count its tokens to check two things: whether it fits within the model's context window, and what the API call will cost.

Reference: 
- https://developers.openai.com/
- https://developers.openai.com/api/docs/models/gpt-5.4-mini
- https://platform.openai.com/tokenizer


In [ ]:
import tiktoken

# gpt-5.4-mini uses the o200k_base encoding (same family as GPT-4o/GPT-5)
encoding = tiktoken.get_encoding("o200k_base")

prompt_tokens = len(encoding.encode(prompt))

print(f"Prompt tokens: {prompt_tokens:,}")

## Estimate the Request Cost

With the token count from above, we can calculate how much this API call will cost before we send it.

In [ ]:
input_price_per_1m_tokens = 0.75  # $ per 1M input tokens
prompt_cost = (prompt_tokens / 1_000_000) * input_price_per_1m_tokens

print(f"Estimated prompt cost: ${prompt_cost:.4f}")

## Summarize the Responsibilities

Now we send the combined prompt to the model and display the response.

The model reads all job descriptions at once and returns the responsibilities that appear most often across the postings.

In [ ]:
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt,
)

display(Markdown(response.output_text))